In [1]:
from api_setup import setup
import json

client, model_id = setup()

In [2]:
def add_user_message(messages, content):
    if isinstance(content, str):
        user_message = {"role": "user", "content": [{"text": content}]}
    else:
        user_message = {"role": "user", "content": content}
    messages.append(user_message)


def add_assistant_message(messages, content):
    if isinstance(content, str):
        assistant_message = {"role": "assistant", "content": [{"text": content}]}
    else:
        assistant_message = {"role": "assistant", "content": content}
    messages.append(assistant_message)


def chat(messages, system=None, tools=None, **kwargs):
    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {
            "temperature": kwargs.get("temperature", 1.0),
            "stopSequences": kwargs.get("stop_sequences", []),
        },
    }

    if system:
        params["system"] = [{"text": system}]

    if tools:
        params["toolConfig"] = {"tools": tools, "toolChoice": {"auto": {}}}

    response = client.converse(**params)
    parts = response["output"]["message"]["content"]

    return {
        "parts": parts,
        "stop_reason": response["stopReason"],
        "text": "\n".join([p["text"] for p in parts if "text" in p])
    }

In [9]:
article_details_schema = {
    "toolSpec": {
        "name": "article_details",
        "description": "Extracts key information from an article",
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {
                    "title": {
                        "type": "string",
                        "description": "The title of the article"
                    },
                    "author": {
                        "type": "string", 
                        "description": "The author's name"
                    },
                    "topics": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "List of key topics mentioned"
                    }
                },
                "required": ["title", "author", "topics"]
            }
        }
    }
}

# Flexible tool extraction. Replaces complex tool schema for data extraction
to_json_schema = {
    "toolSpec": {
        "name": "to_json",
        "description": """
            This tool processes any JSON data and can be used for generating 
            structured content. Transforming information, or creating any 
            JSON-based output needed for your task.
        """,
        "inputSchema": {
            "json": {
                "type": "object",
                "additionalProperties": True
            }
        }
    }
}

In [11]:
messages = []

add_user_message(
    messages,
    "Write a one-paragraph scholarly article about computer science. Include a title and author name",
)

result = chat(messages)

add_assistant_message(messages, result["text"])

result["text"]

'# The Evolution of Machine Learning: From Statistical Methods to Deep Neural Networks\n\n**By Dr. Alexandra Chen**\n\nMachine learning has undergone a profound transformation over the past three decades, evolving from classical statistical approaches such as linear regression and decision trees into sophisticated deep learning architectures capable of processing vast, unstructured datasets with remarkable efficacy. This paradigm shift, catalyzed by the exponential growth in computational power, the proliferation of big data, and algorithmic innovations including backpropagation refinements and convolutional neural networks, has fundamentally altered the landscape of artificial intelligence applications across domains ranging from natural language processing to computer vision. Contemporary deep learning models, particularly transformer-based architectures like BERT and GPT, have achieved state-of-the-art performance on numerous benchmark tasks, yet this success has also precipitated c

In [6]:
messages = []
add_user_message(messages, f"""
Analyze the article below and extract key data. Then call the article_details tool.

<article_text>
{result['text']}
</article_text>
""")

result = chat(messages, tools=[article_details_schema], tool_choice="article_details")

In [7]:
print(result)

{'parts': [{'toolUse': {'toolUseId': 'tooluse_8z7Xocec2ZJEIzM6zuoe6Z', 'name': 'article_details', 'input': {'title': 'The Evolution of Algorithmic Complexity: From Theoretical Foundations to Practical Applications', 'author': 'Dr. Margaret Chen, Department of Computer Science, Stanford University', 'topics': ['algorithmic complexity', 'Big O notation', 'complexity analysis', 'algorithm performance', 'cryptography', 'machine learning', 'distributed systems', 'quantum computing', 'complexity classes', 'BQP', 'computational limits', 'artificial intelligence']}, 'type': 'tool_use'}}], 'stop_reason': 'tool_use', 'text': ''}


In [13]:
messages = []
add_user_message(messages, f"""
Analyze the article below and extract key data. Then call the to_json tool.

<article_text>
{result["text"]}
</article_text>

When you call to_json, pass in the following structure:
{{
    "title": str # title of the article,
    "author": str # author of the article,
    "topics": List[str] # List of topics mentioned in the article
    "num_topics": int # Number of topics mentioned
}}
""")

flexible_result = chat(messages, tools=[to_json_schema], tool_choice="to_json")

print(flexible_result)

{'parts': [{'toolUse': {'toolUseId': 'tooluse_TTWzyMFryW1toYt8lOu9uv', 'name': 'to_json', 'input': {'title': 'The Evolution of Machine Learning: From Statistical Methods to Deep Neural Networks', 'author': 'Dr. Alexandra Chen', 'topics': ['machine learning', 'statistical methods', 'linear regression', 'decision trees', 'deep learning', 'neural networks', 'computational power', 'big data', 'backpropagation', 'convolutional neural networks', 'natural language processing', 'computer vision', 'transformer-based architectures', 'BERT', 'GPT', 'interpretability', 'computational requirements', 'ethical concerns', 'bias and fairness', 'explainable artificial intelligence (XAI)', 'energy-efficient models', 'responsible AI', 'transparency', 'accountability'], 'num_topics': 24}, 'type': 'tool_use'}}], 'stop_reason': 'tool_use', 'text': ''}
